## NB_hospital_daily_summary
Builds **`<catalog>.gold.hospital_daily_summary`** as defined in `Gold/sttm_silver_to_gold.xlsx` (sheet `hospital_daily_summary`).

* **Grain:** one row per hospital per calendar day (days with at least one lab test)
* **Driving table:** `<catalog>.silver.lab_results` grouped by `hospital_id`, `test_date`
* **Joins (LEFT):** `silver.hospitals` on `hospital_id`; `silver.calendar` on `date_key = yyyyMMdd(test_date)`
* **Window:** `LAG(total_tests)` partitioned by `hospital_id`, ordered by `summary_date` (previous day with tests)
* **Filters:** none
* **Load type:** FULL, `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
# Catalog placeholder from the STTM (<catalog>); defaults to adb_caresync
dbutils.widgets.text("catalog", "adb_caresync")
catalog = dbutils.widgets.get("catalog").strip()

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.hospital_daily_summary"

print("=" * 80)
print("[STEP 1] Notebook parameters")
print(f"  catalog       : {catalog}")
print(f"  source schema : {source_schema}")
print(f"  target table  : {target_table}")
print("=" * 80)

In [0]:
# daily_agg: one row per hospital per test_date; with_lag adds the previous day's volume
hospital_daily_summary_sql = f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH daily_agg AS (
  SELECT
    hospital_id,
    test_date                                               AS summary_date,
    COUNT(*)                                                AS total_tests,
    COUNT(DISTINCT patient_id)                              AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                               AS active_doctor_count,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count
  FROM {source_schema}.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily_agg d
)
SELECT
  CAST(w.hospital_id AS INT)                                                  AS hospital_id,
  CAST(h.hospital_name AS STRING)                                             AS hospital_name,
  CAST(w.summary_date AS DATE)                                                AS summary_date,
  CAST(c.day_name AS STRING)                                                  AS day_name,
  CAST(c.is_weekend AS BOOLEAN)                                               AS is_weekend,
  CAST(w.total_tests AS INT)                                                  AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                       AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                          AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                          AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                          AS critical_test_count,
  CAST(ROUND(w.abnormal_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(w.critical_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                         AS prev_day_total_tests,
  CAST(
    CASE
      WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
      ELSE ROUND((w.total_tests - w.prev_day_total_tests) * 100.0 / w.prev_day_total_tests, 2)
    END AS DECIMAL(6,2))                                                      AS test_volume_change_pct
FROM with_lag w
LEFT JOIN {source_schema}.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN {source_schema}.calendar c
  ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)
"""

print(f"[STEP 2] Building {target_table}")
print("  Query prepared:")
print(hospital_daily_summary_sql)
spark.sql(hospital_daily_summary_sql)
print(f"  {target_table} created successfully")
print("=" * 80)

In [0]:
# Validate grain (one row per hospital per day) and that every lab test is counted once
validation_sql = f"""
SELECT
  (SELECT COUNT(DISTINCT hospital_id, test_date) FROM {source_schema}.lab_results) AS source_hospital_days,
  COUNT(*)                                                  AS target_rows,
  COUNT(DISTINCT hospital_id, summary_date)                 AS distinct_hospital_days,
  SUM(total_tests)                                          AS total_tests,
  (SELECT COUNT(*) FROM {source_schema}.lab_results)        AS source_lab_results,
  COUNT(CASE WHEN hospital_name IS NULL THEN 1 END)         AS missing_hospital_name,
  COUNT(CASE WHEN day_name IS NULL THEN 1 END)              AS missing_calendar_match
FROM {target_table}
"""
print("[STEP 3] Validating")
print("  Query prepared:")
print(validation_sql)
v = spark.sql(validation_sql).first()
for k, val in v.asDict().items():
    print(f"  {k:<24}: {val}")

if not (v.source_hospital_days == v.target_rows == v.distinct_hospital_days):
    raise ValueError("Grain check failed: expected one row per hospital per day")
if v.total_tests != v.source_lab_results:
    raise ValueError("Reconciliation failed: total_tests does not match silver.lab_results row count")
print("  Grain and reconciliation checks passed")
print("=" * 80)

display(spark.table(target_table).orderBy("hospital_id", "summary_date").limit(10))

In [0]:
# Return the gold table record count to the caller (ADF / parent notebook)
record_count = spark.table(target_table).count()
print(f"[STEP 4] Record count for {target_table}: {record_count}")
dbutils.notebook.exit(str(record_count))